# Recipe relaxation gates, solve policies, and overrides

Global `RelaxationSettings` describe the solver for the whole run. A
`SolvePolicy` is the rule one recipe gate must satisfy, while
`RelaxationOverrides` temporarily changes how that gate approaches its
targets. Both are passed to `Recipe.solve(policy, overrides)`. This
supports contact-first assembly followed by strict final bending
cleanup.

In [ ]:
# Policies control acceptance; overrides control the path there.
import tangle
from tangle.units import mm, um

## Every `SolvePolicy` field

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `name` | Stage label used in reports and errors. | string |
| `target_penetration` | Penetration residual the stage solver works toward. | m |
| `target_curvature_ratio` | Curvature ratio the stage solver works toward. | ratio |
| `max_penetration` | Penetration required to accept the stage; defaults to `target_penetration`. | m |
| `max_curvature_ratio` | Curvature ratio required to accept the stage; defaults to `target_curvature_ratio`. | ratio |
| `hard_penetration` | Whether a penetration miss fails the stage (`True`) or is only reported. | boolean |
| `hard_curvature` | Whether a curvature miss fails the stage (`True`) or is only reported. | boolean |
| `max_iterations` | Stage-specific iteration budget. | count |
| `on_budget_exhausted` | Behavior when the budget runs out before the targets are met. | `"fail"` or `"continue_if_hard_ok"` |

In [ ]:
# Read defaults from the compiled extension instead of duplicating
# them in documentation that could become stale.
default_policy = tangle.SolvePolicy()
fields = ['name', 'target_penetration', 'target_curvature_ratio', 'max_penetration', 'max_curvature_ratio', 'hard_penetration', 'hard_curvature', 'max_iterations', 'on_budget_exhausted']
{name: getattr(default_policy, name) for name in fields}

A policy usually needs only its targets. `max_penetration` and
`max_curvature_ratio` default to the targets, both limits are hard,
and an exhausted budget fails the recipe. Loosen acceptance only
where a stage should tolerate it.

In [ ]:
# This assembly stage must resolve contact but only reports curvature,
# accepting ratios up to 5 so bending cannot block deposition.
contact_first = tangle.SolvePolicy(
    "contact-first settling",
    target_penetration=0.1 * um,
    max_penetration=0.2 * um,
    target_curvature_ratio=5.0,
    hard_curvature=False,
    max_iterations=10_000,
)
# A strict final gate needs only its targets; the acceptance limits
# default to them and both are hard.
final = tangle.SolvePolicy(
    "final",
    target_penetration=0.1 * um,
    target_curvature_ratio=1.02,
    max_iterations=5_000,
)
# Continue past an exhausted budget when every hard limit is met.
lenient = final.replace(
    name="final (lenient)",
    hard_curvature=False,
    on_budget_exhausted="continue_if_hard_ok",
)
[final.max_penetration, final.max_curvature_ratio, final.hard_penetration, lenient.on_budget_exhausted]

## Every `RelaxationOverrides` field

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `motion_model` | Temporary fiber motion model. | string or `None` |
| `correction_fraction` | Temporary contact correction fraction. | float or `None` |
| `contact_aggregation` | Temporary contact aggregation rule. | string or `None` |
| `stretch_stiffness` | Temporary rest-length stiffness. | float or `None` |
| `bend_stiffness` | Temporary rest-bend stiffness. | float or `None` |
| `curvature_limit_stiffness` | Temporary hard-curvature stiffness. | float or `None` |
| `constraint_iterations` | Temporary constraint sweep count. | integer or `None` |
| `curvature_cleanup_sweeps` | Temporary curvature cleanup count. | integer or `None` |

A field left as `None` keeps the global setting. Three presets capture
the common staging choices, and any field can follow the preset name
as a keyword:

- `contact_first`: full contact corrections with loose, cheap mechanics
  (no rest bending, weak curvature limit, one constraint sweep).
- `contact_cleanup`: full contact corrections with strong stretch and
  curvature projection and many sweeps.
- `curvature_cleanup`: small contact corrections with full curvature
  projection and many cleanup sweeps, for pulling bends back inside
  the admissible limit.

In [ ]:
# Presets are ordinary RelaxationOverrides; print them to see exactly
# which fields each one sets.
for preset in ("contact_first", "contact_cleanup", "curvature_cleanup"):
    print(repr(tangle.RelaxationOverrides.preset(preset)))
# Keywords after the preset name adjust individual fields.
cleanup = tangle.RelaxationOverrides.preset("curvature_cleanup", curvature_cleanup_sweeps=32)
# Hand-written overrides set only the fields they name.
softer_bending = tangle.RelaxationOverrides(bend_stiffness=0.1, contact_aggregation="deepest_only")
softer_bending

In [ ]:
# Overrides apply only to the solve() they are passed to; the global
# settings return for later operations.
recipe = tangle.Recipe(tangle.Cell([1 * mm, 1 * mm, 1 * mm]))
# These calls represent distinct amounts or acceptance conditions.
recipe.relax_for(100)  # fixed work; no acceptance gate
recipe.relax_until_converged(max_iterations=2_000)  # default hard gate
recipe.solve(contact_first, tangle.RelaxationOverrides.preset("contact_first"))
recipe.solve(final, cleanup)
# Bend limits can change between stages, by material name or object.
recipe.set_min_bend_radius("fiber", 50 * um)
print(*recipe.operations(), sep="\n")

`settle_targets(tolerance=, max_iterations=)` is the gate for fibers
held on layer-placement or needle targets; tutorial 10 uses it.

## When a stage fails

`Recipe.run()` raises `tangle.RecipeError` (a `RuntimeError`) when an
operation cannot meet a hard limit within its budget, or fails
validation. The exception says which step failed: `operation_index`
(zero-based position in `recipe.operations()`), `operation` (its
description), `iteration` (the solver iteration at failure), and
`reason`.

In [ ]:
# Deliberately give a stage one iteration to show the error fields.
# Running it takes a few seconds on the CPU backend, so it is opt-in.
RUN_SOLVER = False
if RUN_SOLVER:
    cell = tangle.Cell([1 * mm, 1 * mm, 1 * mm])
    fibers = tangle.generate_fiber_pair_crossing(
        cell,
        material=tangle.Material("fiber", diameter=19 * um),
        length=0.8 * mm,
        axis_separation=10 * um,
    )
    failing = tangle.Recipe(cell)
    failing.insert(fibers)
    failing.solve(final.replace(name="too short", max_iterations=1))
    try:
        failing.run(tangle.RelaxationSettings(backend="cpu"))
    except tangle.RecipeError as error:
        print(error.operation_index, error.operation, error.iteration)
        print(error.reason)